In [ ]:
!pip -q install "langchain==1.3.14" "langchain-google-genai==4.2.7" "gradio==6.20.0" "pydantic>=2.13,<3"

# **1ST STEP**

In [ ]:
import os
import getpass
import json
from typing import Literal, Optional
import gradio as gr
from pydantic import BaseModel, Field, ValidationError
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

# Load Gemini API Key
api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get("GOOGLE_API_KEY")
except Exception:
    api_key = None

if not api_key:
    api_key = getpass.getpass("Enter your Google Gemini API key: ")

if not api_key or not api_key.strip():
    raise ValueError("A valid GOOGLE_API_KEY is required.")

os.environ["GOOGLE_API_KEY"] = api_key.strip()

# Initialize AI Model
MODEL_NAME = "gemini-3.1-flash-lite"
llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    thinking_level="low",
)
print(f"Car Recommendation Agent Ready | Model: {MODEL_NAME}")

Car Recommendation Agent Ready | Model: gemini-3.1-flash-lite


# **2ND** **STEP**

In [ ]:
class CarFacts(BaseModel):
    """Information extracted from the user's car request."""
    budget_sar: Optional[float] = Field(
        default=None, ge=0, description="Total budget in Saudi Riyals (SAR)."
    )
    body_style: Optional[str] = Field(
        default=None,
        description="Type of vehicle (e.g., SUV, Sedan, Sport, Coupe, Hatchback, Pickup Truck, Convertible, Crossover, Minivan, Wagon)."
    )
    fuel_preference: Literal["gasoline", "hybrid", "electric", "diesel", "unspecified"] = Field(
        default="unspecified", description="Preferred fuel or powertrain type."
    )
    passengers: int = Field(
        default=1, ge=1, le=12, description="Number of passengers or family members."
    )
    primary_use: list[str] = Field(
        default_factory=list, description="Primary usage (e.g., city commute, long highway trips, track/performance, off-roading, family hauling)."
    )
    key_features: list[str] = Field(
        default_factory=list, description="Important features requested (e.g., safety, fuel efficiency, luxury, speed, tech, cargo space)."
    )
    missing_information: list[str] = Field(
        default_factory=list, description="Essential car buying details not supplied by the user."
    )

class CarRecommendation(BaseModel):
    """Recommendation decision produced from extracted facts."""
    recommended_car: str = Field(description="Primary recommended vehicle make, model, and trim if applicable.")
    alternative_car: Optional[str] = Field(default=None, description="Alternative competitor vehicle in the same segment and budget.")
    short_rationale: str = Field(description="Short explanation of why this car fits the user's criteria.")
    budget_fit: Literal["comfortable", "possible", "tight", "unknown"] = Field(
        description="Assessment of how well the vehicle fits the given budget."
    )
    assumptions: list[str] = Field(default_factory=list, description="Assumptions made during recommendation.")

class CarSpecification(BaseModel):
    """Detailed specifications and evaluation for the final report."""
    car_name: str
    category: str
    pros: list[str] = Field(description="Key advantages and highlights.")
    cons: list[str] = Field(description="Key drawbacks or points to consider.")
    estimated_price_range: str = Field(description="Estimated market price range in SAR.")
    verdict: str = Field(description="Final summary evaluation for the user.")

class FinalCarPlan(BaseModel):
    """Final comprehensive report delivered to the user."""
    title: str
    recommended_car: str
    summary: str
    car_details: CarSpecification
    alternative_car: Optional[str] = None
    assumptions: list[str] = Field(default_factory=list)
    limitations: list[str] = Field(default_factory=list)

# **3ED STEP**

In [ ]:
CAR_GUIDE = """
Use this category matching guide for recommendations:

- Full-Size & Family SUVs (7-8 Passengers):
  * Toyota Land Cruiser, Nissan Patrol, Hyundai Palisade, Kia Telluride, Chevrolet Tahoe, Ford Expedition.
  * Best for: Large families, road trips, off-roading, towing, and max comfort.

- Compact & Mid-Size Crossovers / SUVs:
  * Toyota RAV4, Honda CR-V, Hyundai Tucson, Kia Sportage, Mazda CX-5, Haval H6, Geely Monjaro.
  * Best for: Small families, daily commuting, balanced fuel economy, and higher drive height.

- Sedans (Compact, Mid-Size, Full-Size):
  * Toyota Camry, Honda Accord, Hyundai Elantra/Sonata, Nissan Altima, Lexus ES, BMW 5 Series.
  * Best for: Daily highway driving, smooth rides, passenger comfort, and great fuel efficiency.

- Sports Cars & Performance Coupes:
  * Ford Mustang, Chevrolet Corvette, Toyota GR Supra, Porsche 911 / Cayman, BMW M4, Nissan Z.
  * Best for: Driving enthusiasts, speed, high performance, track days, and aggressive styling.

- Hatchbacks:
  * Volkswagen Golf / GTI, Toyota Yaris Hatchback, Peugeot 208, MINI Cooper, Hyundai i30.
  * Best for: Easy city parking, compact footprint, agile handling, and practical cargo accessibility.

- Pickup Trucks:
  * Toyota Hilux, Ford F-150, RAM 1500, Chevrolet Silverado, Isuzu D-Max.
  * Best for: Heavy towing, hauling, off-road durability, and utility work.

- Luxury Cars & Luxury SUVs:
  * Mercedes-Benz S-Class / GLE, BMW 7 Series / X5, Audi A8 / Q7, Lexus LX600, Genesis G90.
  * Best for: Executive comfort, top-tier luxury, advanced tech, and prestige.

- Electric Vehicles (EV) & Hybrids:
  * Tesla Model 3 / Model Y, BYD Seal / Atto 3, Toyota Camry Hybrid, Hyundai Ioniq 5.
  * Best for: Zero/low fuel costs, instant acceleration, tech enthusiasts, and eco-friendly driving.

- Minivans & Passenger Vans:
  * Toyota Sienna, Kia Carnival, Honda Odyssey.
  * Best for: Maximum interior passenger space, sliding doors, and ultimate family practicality.

- Convertible / Roadster:
  * Mazda MX-5 Miata, BMW Z4, Porsche 718 Boxster, Ford Mustang Convertible.
  * Best for: Open-air driving experience, stylish weekend cruising.

Important limitations:
- Market prices are estimates only and vary by dealer, options, and market availability.
- Always recommend test-driving at an authorized local dealership before purchase.
"""

# **4TH STEP**

In [ ]:
extract_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are the requirement-extraction component of Car Recommendation Agent.
Extract only information present in the user's request.

Rules:
- Do not invent budget, body style, features, car brand, or passenger counts.
- If budget is missing, return null and add "budget" to missing_information.
- If passengers count is missing, default to 1 and add "passengers count" to missing_information.
- Map requested vehicle body style to standard vehicle segments (SUV, Sedan, Sport, Coupe, Hatchback, Pickup Truck, Crossover, Convertible, Minivan, Wagon).
- Return output matching the schema.
""",
    ),
    ("human", "Car Request:\n{request}"),
])

extract_chain = extract_prompt | llm.with_structured_output(
    schema=CarFacts.model_json_schema(), method="json_schema"
)

def extract_car_facts(request: str) -> CarFacts:
    raw_result = extract_chain.invoke({"request": request})
    return CarFacts.model_validate(raw_result)

# Stage 2: Recommend matching car based on guide
recommend_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are the recommendation component of Car Recommendation Agent.
Use the extracted requirements and the car knowledge guide.

Rules:
- Recommend one primary car model (including brand and segment) matching budget, passenger count, body style, and usage.
- Suggest one strong alternative competitor in the same segment.
- Provide a clear, concise rationale for the choice.
- State all assumptions clearly.
""",
    ),
    (
        "human",
        """
Extracted Car Facts:
{facts_json}

Car Guide:
{car_guide}
""",
    ),
])

recommend_chain = recommend_prompt | llm.with_structured_output(
    schema=CarRecommendation.model_json_schema(), method="json_schema"
)

def recommend_car(facts: CarFacts) -> CarRecommendation:
    raw_result = recommend_chain.invoke({
        "facts_json": facts.model_dump_json(indent=2),
        "car_guide": CAR_GUIDE,
    })
    return CarRecommendation.model_validate(raw_result)

# Stage 3: Generate detailed specification & evaluation report
details_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are the reporting component of Car Recommendation Agent.
Build a comprehensive car assessment report using the validated car facts and recommendation.

Rules:
- Provide specific pros and cons for the recommended car.
- Provide an estimated market price range in SAR.
- Give a clear final verdict.
- Clearly state assumptions and limitations regarding price and availability.
""",
    ),
    (
        "human",
        """
Car Facts:
{facts_json}

Recommendation:
{recommendation_json}

Car Guide:
{car_guide}
""",
    ),
])

details_chain = details_prompt | llm.with_structured_output(
    schema=FinalCarPlan.model_json_schema(), method="json_schema"
)

def create_car_plan(facts: CarFacts, recommendation: CarRecommendation) -> FinalCarPlan:
    raw_result = details_chain.invoke({
        "facts_json": facts.model_dump_json(indent=2),
        "recommendation_json": recommendation.model_dump_json(indent=2),
        "car_guide": CAR_GUIDE,
    })
    return FinalCarPlan.model_validate(raw_result)

# **5TH STEP**

In [ ]:
def run_car_recommendation_agent(request: str) -> dict:
    """Run the complete Extract -> Recommend -> Report pipeline."""
    if not isinstance(request, str) or not request.strip():
        raise ValueError("Please provide a valid car requirement request.")

    facts = extract_car_facts(request.strip())
    recommendation = recommend_car(facts)
    final_plan = create_car_plan(facts, recommendation)

    return final_plan.model_dump()

def generate_car_plan_from_fields(
    description: str,
    budget: float,
    body_style: str,
    fuel_preference: str,
    passengers: float
) -> dict:
    """Combine Gradio input fields into a structured request and run pipeline."""
    combined_request = f"""
Additional preferences and description: {description}
Budget: {budget} SAR
Preferred Body Style / Vehicle Type: {body_style}
Preferred Powertrain / Fuel Type: {fuel_preference}
Number of Passengers: {int(passengers) if passengers else 1}
"""
    try:
        return run_car_recommendation_agent(combined_request)
    except Exception as error:
        return {"error": str(error)}

# **6TH STEP**

In [ ]:

def generate_formatted_car_report(
    description: str,
    budget: float,
    body_style: str,
    fuel_preference: str,
    passengers: float
) -> str:
    """Runs the pipeline and formats the JSON response into a clean Markdown layout."""
    combined_request = f"""
Additional preferences and description: {description}
Budget: {budget} SAR
Preferred Body Style / Vehicle Type: {body_style}
Preferred Powertrain / Fuel Type: {fuel_preference.lower()}
Number of Passengers: {int(passengers) if passengers else 1}
"""
    try:
        data = run_car_recommendation_agent(combined_request)

        details = data.get("car_details", {})
        pros_list = "\n".join([f"  • {p}" for p in details.get("pros", [])])
        cons_list = "\n".join([f"  • {c}" for c in details.get("cons", [])])
        assumptions_list = "\n".join([f"  • {a}" for a in data.get("assumptions", [])])

        formatted_markdown = f"""
# 🚙 Recommended Vehicle: {data.get('recommended_car', 'N/A')}

> **Summary:** {data.get('summary', '')}

---

### 📋 Vehicle Overview & Specs
* 🏷️ **Category:** {details.get('category', 'N/A')}
* 💰 **Estimated Price Range:** `{details.get('estimated_price_range', 'N/A')}`
* 🔄 **Alternative Choice:** `{data.get('alternative_car', 'None')}`

---

### ✨ Pros & Highlights
{pros_list}

---

### 💡 Points to Consider
{cons_list}

---

### 🏁 Final Verdict
{details.get('verdict', '')}

---

<details>
<summary><b>ℹ️ Click to view assumptions & notes</b></summary>

{assumptions_list}
</details>
"""
        return formatted_markdown

    except Exception as error:
        return f"⚠️ **Error:** {str(error)}"


button_group_css = """
/* Dark Background for High Contrast */
body, .gradio-container {
    background-color: #0f172a !important;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
}

/* Main Cards Containers */
.input-card-box, .output-card-box {
    background-color: #1e293b !important;
    border: 2px solid #334155 !important;
    border-radius: 16px !important;
    padding: 24px !important;
    box-shadow: 0 10px 25px -5px rgba(0, 0, 0, 0.5) !important;
}

/* Headings with Purple Badge Styling */
.input-card-box h2, .output-card-box h2 {
    color: #ffffff !important;
    background-color: #6366f1 !important;
    display: inline-block !important;
    padding: 8px 18px !important;
    border-radius: 10px !important;
    font-size: 20px !important;
    font-weight: bold !important;
    margin-bottom: 15px !important;
    box-shadow: 0 4px 12px rgba(99, 102, 241, 0.4) !important;
}

/* تخصيص عناوين الحقول الرئيسية فقط بدون التأثير على أزرار الوقود */
.input-card-box > label span,
.input-card-box .gradio-dropdown span,
.input-card-box .gradio-slider span,
.input-card-box .gradio-number span,
.input-card-box .gradio-textbox span {
    background-color: #6366f1 !important;
    color: #ffffff !important;
    padding: 6px 12px !important;
    border-radius: 8px !important;
    font-weight: bold !important;
    font-size: 15px !important;
    display: inline-block !important;
    margin-bottom: 8px !important;
}

/* Slider Number Display */
.gradio-slider input[type="number"] {
    background-color: #0f172a !important;
    color: #38bdf8 !important;
    font-weight: bold !important;
    font-size: 16px !important;
    border: 1px solid #38bdf8 !important;
    border-radius: 6px !important;
}

/* Radio Button Options Layout */
.fuel-radio-group .options {
    display: flex !important;
    flex-wrap: wrap !important;
    gap: 8px !important;
}

/* الشكل التلقائي لزر الوقود غير المختار */
.fuel-radio-group label {
    background-color: #334155 !important;
    color: #ffffff !important;
    border: 1px solid #475569 !important;
    border-radius: 8px !important;
    padding: 8px 16px !important;
    cursor: pointer !important;
    font-weight: 600 !important;
    margin: 0 !important;
    transition: all 0.2s ease !important;
}

/* إزالة أي خلفية بنفسجية ملتصقة بالنص الداخلي لأزرار الوقود */
.fuel-radio-group label span {
    background-color: transparent !important;
    box-shadow: none !important;
    padding: 0 !important;
    margin: 0 !important;
    color: inherit !important;
}

.fuel-radio-group label:hover {
    background-color: #475569 !important;
}

/* التنسيق البنفسجي للزر المختار فقط بالكامل */
.fuel-radio-group label.selected, .fuel-radio-group label:has(input:checked) {
    background-color: #6366f1 !important;
    color: #ffffff !important;
    border-color: #818cf8 !important;
    box-shadow: 0 4px 12px rgba(99, 102, 241, 0.5) !important;
    font-weight: 800 !important;
}

.fuel-radio-group input[type="radio"] {
    display: none !important;
}

/* Primary Submit Button */
#submit-btn {
    background: linear-gradient(90deg, #6366f1, #8b5cf6) !important;
    color: #ffffff !important;
    font-size: 18px !important;
    font-weight: 800 !important;
    border-radius: 12px !important;
    padding: 14px !important;
    margin-top: 15px !important;
    border: none !important;
    box-shadow: 0 4px 15px rgba(99, 102, 241, 0.5) !important;
    transition: all 0.2s ease-in-out;
}

#submit-btn:hover {
    transform: translateY(-2px);
    box-shadow: 0 6px 20px rgba(99, 102, 241, 0.7) !important;
}
"""

with gr.Blocks(title="Car Recommendation Agent", theme=gr.themes.Soft(), css=button_group_css) as demo:

    with gr.Row():
        gr.HTML(
            """
            <div style="background: linear-gradient(135deg, #4f46e5, #7c3aed); padding: 20px; border-radius: 16px; text-align: center; margin-bottom: 20px; box-shadow: 0 10px 20px rgba(0,0,0,0.3);">
                <h1 style="color: #ffffff; margin: 0; font-size: 30px; font-weight: 800;">🚗 Car Recommendation Agent</h1>
                <p style="color: #e0e7ff; margin-top: 6px; font-size: 16px;">Smart, Fast & Personalized AI Vehicle Matching</p>
            </div>
            """
        )

    with gr.Row(equal_height=False):

        with gr.Column(scale=1, elem_classes=["input-card-box"]):
            gr.Markdown("## ⚙️ 1. Define Preferences")

            description_input = gr.Textbox(
                lines=3,
                label="📝 Extra Details & Special Requests",
                placeholder="Type any specific needs here (e.g., High safety rating, ventilated seats, quiet highway cabin...)"
            )

            budget_input = gr.Slider(
                minimum=0,
                maximum=500000,
                step=25000,
                value=150000,
                label="💰 Budget Limit (SAR)"
            )

            body_style_input = gr.Dropdown(
                choices=[
                    "SUV / Family",
                    "Crossover",
                    "Sedan",
                    "Sport / Performance",
                    "Coupe",
                    "Hatchback",
                    "Pickup Truck",
                    "Convertible",
                    "Minivan / Van",
                    "Wagon",
                    "Unspecified / Any"
                ],
                value="SUV / Family",
                label="🚘 Vehicle Body Style"
            )

            fuel_preference_input = gr.Radio(
                choices=["Gasoline", "Hybrid", "Electric", "Diesel", "Unspecified"],
                value="Gasoline",
                label="⚡ Powertrain / Fuel Type",
                elem_classes=["fuel-radio-group"]
            )

            passenger_input = gr.Number(
                value=5,
                precision=0,
                label="👥 Passenger Count"
            )

            submit_btn = gr.Button("🚀 Generate Recommendation", variant="primary", elem_id="submit-btn")

        with gr.Column(scale=1.2, elem_classes=["output-card-box"]):
            gr.Markdown("## 📊 2. Recommendation Report")
            output_report = gr.Markdown(
                value="Fill in your requirements on the left and click **Generate Recommendation** to display your custom report."
            )

    submit_btn.click(
        fn=generate_formatted_car_report,
        inputs=[description_input, budget_input, body_style_input, fuel_preference_input, passenger_input],
        outputs=output_report
    )

demo.launch(share=True, debug=True)

/tmp/ipykernel_4188/1686605505.py:186: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Car Recommendation Agent", theme=gr.themes.Soft(), css=button_group_css) as demo:
/usr/local/lib/python3.13/dist-packages/gradio/layouts/column.py:59: UserWarning: 'scale' value should be an integer. Using 1.2 will cause issues.
  warnings.warn(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1562270d4589e2e350.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
